In [2]:
import wandb
import pandas as pd
import numpy as np

import json

api = wandb.Api()

from helpers import *

## IAUC DAUC

In [3]:
iauc_dauc_runs = api.runs(
    "pasqualedem/affex",
    filters={
        "$and": [
            {"config.metric.n_steps": 75},
            {"tags": {"$nin": ["Reverse"]}},
            {"$or": [
                {'config.dataset.datasets.val_deepglobe_N1K5.name': "deepglobe"},
                {'config.dataset.datasets.val_lung_N1K5.name': "lung"},
                {'config.dataset.datasets.val_isic_N1K5.name': "isic"},
            ]}
        ]
    }
)

In [4]:
iauc_dauc_runs_df = get_runs_df(iauc_dauc_runs)
len(iauc_dauc_runs_df)

12

In [5]:
res_df = iauc_dauc_runs_df.copy()

dataset_names = ["val_lung_N1K5", "val_isic_N1K5", "val_deepglobe_N1K5"]

dataset_cols = [f'dataset.datasets.{name}.name' for name in dataset_names]
shots_cols = [f'dataset.datasets.{name}.n_shots' for name in dataset_names]
res_df["Shots"] = res_df[shots_cols].fillna("").astype(str).sum(axis=1).astype(float).astype(int)
res_df["Dataset"] = res_df[dataset_cols].fillna("").sum(axis=1) 

iauc_cols = [f"{name}_iauc" for name in dataset_names]
dauc_cols = [f"{name}_dauc" for name in dataset_names]
assert res_df[iauc_cols].isna().any(axis=1).all(), "More than one dataset is present in the same run, please check the dataset columns."
assert res_df[dauc_cols].isna().any(axis=1).all(), "More than one dataset is present in the same run, please check the dataset columns."

res_df = res_df.rename(columns=renamings_dict)

res_df["Explanation Method"] = res_df["Explanation Method"].replace(value_renamings_dict)
res_df["Dataset"] = res_df["Dataset"].replace(value_renamings_dict)
res_df["Model"] = res_df["Model"].replace(value_renamings_dict)


res_df["IAUC"] = res_df[iauc_cols].fillna(0).sum(axis=1)
res_df["DAUC"] = res_df[dauc_cols].fillna(0).sum(axis=1)

res_df["Diff."] = res_df["IAUC"] - res_df["DAUC"]

cols = ["Model", "Explanation Method", "Dataset", "Shots"]
res_df = res_df[cols + ["IAUC", "DAUC", "Diff."]]

print(res_df)

res_df = res_df.pivot_table(
    index=['Dataset', 'Explanation Method'],
    columns=["Shots"],
    values=['IAUC', 'DAUC', 'Diff.']
).sort_index(axis=1).swaplevel(0, 1, axis=1)


new_cols  = sorted(res_df.columns, key=lambda x: (x[0], metric_order[x[1]]))
res_df = res_df[new_cols]
# Sort by Model and then by Explanation Method using the provided orderings
res_df = res_df.sort_values(
    by=['Dataset', 'Explanation Method'],  # this ensures level names exist and are used
    key=lambda x: x.map(dataset_order) if x.name == 'Dataset' else x.map(method_order)
)

iauc_dauc_df = (res_df*100).round(2)

iauc_dauc_df

     Model     Explanation Method    Dataset  Shots      IAUC      DAUC  \
0   DMTNet  Unmasked AffEx (ours)       ISIC      5  0.790059  0.644435   
1   DMTNet                 Random       ISIC      5  0.739172  0.739680   
2   DMTNet  Unmasked AffEx (ours)       Lung      5  0.902064  0.643075   
3   DMTNet                 Random       Lung      5  0.732055  0.732375   
4   DMTNet  Unmasked AffEx (ours)  Deepglobe      5  0.531181  0.386286   
5   DMTNet                 Random  Deepglobe      5  0.309229  0.309751   
6   DMTNet               Saliency       Lung      5  0.832218  0.721846   
7   DMTNet               Saliency  Deepglobe      5  0.586925  0.430865   
8   DMTNet   Integrated Gradients       ISIC      5  0.725875  0.662099   
9   DMTNet   Integrated Gradients       Lung      5  0.730829  0.747735   
10  DMTNet   Integrated Gradients  Deepglobe      5  0.375315  0.248793   
11  DMTNet               Saliency       ISIC      5  0.809136  0.681742   

       Diff.  
0   0.145

Shots                                5              
                                  IAUC   DAUC  Diff.
Dataset   Explanation Method                        
Deepglobe Random                 30.92  30.98  -0.05
          Saliency               58.69  43.09  15.61
          Integrated Gradients   37.53  24.88  12.65
          Unmasked AffEx (ours)  53.12  38.63  14.49
ISIC      Random                 73.92  73.97  -0.05
          Saliency               80.91  68.17  12.74
          Integrated Gradients   72.59  66.21   6.38
          Unmasked AffEx (ours)  79.01  64.44  14.56
Lung      Random                 73.21  73.24  -0.03
          Saliency               83.22  72.18  11.04
          Integrated Gradients   73.08  74.77  -1.69
          Unmasked AffEx (ours)  90.21  64.31  25.90

In [6]:
from tabulate import tabulate

latex_tab = tabulate(res_df, headers='keys', tablefmt='latex_booktabs', showindex=True)
print(latex_tab)

\begin{tabular}{lrrr}
\toprule
                                        &   (5, 'IAUC') &   (5, 'DAUC') &   (5, 'Diff.') \\
\midrule
 ('Deepglobe', 'Random')                &      0.309229 &      0.309751 &   -0.0005225   \\
 ('Deepglobe', 'Saliency')              &      0.586925 &      0.430865 &    0.15606     \\
 ('Deepglobe', 'Integrated Gradients')  &      0.375315 &      0.248793 &    0.126522    \\
 ('Deepglobe', 'Unmasked AffEx (ours)') &      0.531181 &      0.386286 &    0.144894    \\
 ('ISIC', 'Random')                     &      0.739172 &      0.73968  &   -0.000508215 \\
 ('ISIC', 'Saliency')                   &      0.809136 &      0.681742 &    0.127394    \\
 ('ISIC', 'Integrated Gradients')       &      0.725875 &      0.662099 &    0.0637761   \\
 ('ISIC', 'Unmasked AffEx (ours)')      &      0.790059 &      0.644435 &    0.145624    \\
 ('Lung', 'Random')                     &      0.732055 &      0.732375 &   -0.000320113 \\
 ('Lung', 'Saliency')                   

## IAUC DAUC mIoU

In [7]:
iauc_dauc_runs = api.runs(
    "pasqualedem/affex",
    filters={
        "$and": [
            {"config.metric.n_steps": 75},
            {"tags": {"$nin": ["Reverse"]}},
            {"$or": [
                {'config.dataset.datasets.val_deepglobe_N1K1.name': "deepglobe"},
                {'config.dataset.datasets.val_lung_N1K1.name': "lung"},
                {'config.dataset.datasets.val_isic_N1K1.name': "isic"},
            ]},
            {"config.metric.measure": "miou"}
        ]
    }
)
len(iauc_dauc_runs)

30

In [8]:
iauc_dauc_runs_df = get_runs_df(iauc_dauc_runs)

In [9]:
res_df = iauc_dauc_runs_df.copy()

dataset_names = ["val_lung_N1K1", "val_isic_N1K1", "val_deepglobe_N1K1"]

dataset_cols = [f'dataset.datasets.{name}.name' for name in dataset_names]
shots_cols = [f'dataset.datasets.{name}.n_shots' for name in dataset_names]
res_df["Shots"] = res_df[shots_cols].fillna("").astype(str).sum(axis=1).astype(float).astype(int)
res_df["Dataset"] = res_df[dataset_cols].fillna("").sum(axis=1) 

iauc_cols = [f"{name}_iauc" for name in dataset_names]
dauc_cols = [f"{name}_dauc" for name in dataset_names]

assert res_df[iauc_cols].isna().any(axis=1).all(), "More than one dataset is present in the same run, please check the dataset columns."
assert res_df[dauc_cols].isna().any(axis=1).all(), "More than one dataset is present in the same run, please check the dataset columns."

res_df = res_df.rename(columns=renamings_dict)

res_df["Explanation Method"] = res_df["Explanation Method"].replace(value_renamings_dict)
res_df["Dataset"] = res_df["Dataset"].replace(value_renamings_dict)
res_df["Model"] = res_df["Model"].replace(value_renamings_dict)


res_df["IAUC"] = res_df[iauc_cols].fillna(0).sum(axis=1)
res_df["DAUC"] = res_df[dauc_cols].fillna(0).sum(axis=1)

res_df["Diff."] = res_df["IAUC"] - res_df["DAUC"]

cols = ["Model", "Explanation Method", "Dataset", "Shots"]
res_df = res_df[cols + ["IAUC", "DAUC", "Diff."]]

res_df = res_df.pivot_table(
    index=['Dataset', 'Explanation Method'],
    columns=["Shots"],
    values=['IAUC', 'DAUC', 'Diff.']
).swaplevel(axis=1).sort_index(axis=1)


# new_cols  = sorted(res_df.columns, key=lambda x: (dataset_order[x[0]], metric_order[x[1]]))
# res_df = res_df[new_cols]
# Sort by Model and then by Explanation Method using the provided orderings
res_df = res_df.sort_values(
    by=['Dataset', 'Explanation Method'],  # this ensures level names exist and are used
    key=lambda x: x.map(dataset_order) if x.name == 'Dataset' else x.map(method_order)
)

iauc_dauc_df = (res_df*100).round(2)

iauc_dauc_df

Shots                                1              
                                  DAUC  Diff.   IAUC
Dataset   Explanation Method                        
Deepglobe Random                 15.11   0.65  15.76
          Gaussian Noise Mask    20.51   2.66  23.17
          Saliency               18.33   4.26  22.59
          Integrated Gradients   10.67  10.44  21.11
          Blur IG                19.62   1.07  20.70
          XRAI                   23.67   5.12  28.79
          LIME                   19.63  11.95  31.58
          Unmasked AffEx (ours)  22.06   7.05  29.11
          Masked AffEx (ours)    23.71   5.45  29.16
          Signed AffEx (ours)    21.35   6.61  27.96
ISIC      Random                 42.18  -0.13  42.05
          Gaussian Noise Mask    29.87  16.30  46.17
          Saliency               33.85  11.24  45.09
          Integrated Gradients   39.77   2.50  42.27
          Blur IG                35.23   9.25  44.48
          XRAI                   32.04  11.21  43.25
          LIME                   37.94   3.87  41.81
          Unmasked AffEx (ours)  27.36  15.53  42.90
          Masked AffEx (ours)    28.18  17.81  45.99
          Signed AffEx (ours)    27.86  15.27  43.13
Lung      Random                 43.51   0.21  43.72
          Gaussian Noise Mask    36.65   5.98  42.63
          Saliency               40.11   4.53  44.64
          Integrated Gradients   47.40  -5.33  42.07
          Blur IG                42.06   5.21  47.27
          XRAI                   47.98  -1.19  46.79
          LIME                   48.02  -0.60  47.42
          Unmasked AffEx (ours)  35.44  13.14  48.58
          Masked AffEx (ours)    35.99   8.68  44.67
          Signed AffEx (ours)    33.57  14.12  47.69

## IAUC %

In [10]:
iauc_p_runs = api.runs(
    "pasqualedem/affex",
    filters={
        "$and": [
            {"config.metric.percentage": {"$ne": None}},
            {"tags": {"$nin": ["Reverse"]}},
            {"$or": [
                {'config.dataset.datasets.val_deepglobe_N1K5.name': "deepglobe"},
                {'config.dataset.datasets.val_lung_N1K5.name': "lung"},
                {'config.dataset.datasets.val_isic_N1K5.name': "isic"},
                {'config.dataset.datasets.val_deepglobe_N1K1.name': "deepglobe"},
                {'config.dataset.datasets.val_lung_N1K1.name': "lung"},
                {'config.dataset.datasets.val_isic_N1K1.name': "isic"},
            ]}
        ]
    }
)

In [11]:
iauc_p_runs_df = get_runs_df(iauc_p_runs)
len(iauc_p_runs_df)

186

In [12]:
res_df = iauc_p_runs_df.copy()

dataset_names = ["val_lung_N1K5", "val_isic_N1K5", "val_deepglobe_N1K5", "val_lung_N1K1", "val_isic_N1K1", "val_deepglobe_N1K1"]

dataset_cols = [f'dataset.datasets.{name}.name' for name in dataset_names]
shots_cols = [f'dataset.datasets.{name}.n_shots' for name in dataset_names]
res_df["Shots"] = res_df[shots_cols].fillna("").astype(str).sum(axis=1).astype(float).astype(int)
res_df["Dataset"] = res_df[dataset_cols].fillna("").sum(axis=1) 

iauc_cols = [f"{name}_iauc" for name in dataset_names]
assert res_df[iauc_cols].isna().any(axis=1).all(), "More than one dataset is present in the same run, please check the dataset columns."

res_df = res_df.rename(columns=renamings_dict)

res_df["Explanation Method"] = res_df["Explanation Method"].replace(value_renamings_dict)
res_df["Dataset"] = res_df["Dataset"].replace(value_renamings_dict)
res_df["Model"] = res_df["Model"].replace(value_renamings_dict)


res_df["iAUC"] = res_df[iauc_cols].fillna(0).sum(axis=1)
res_df["Loss"] = res_df["Loss"].fillna(False)

cols = ["Model", "Explanation Method", "Dataset", "Shots", "Percentage", "Loss"]
res_df = res_df[cols + ["iAUC"]]

res_df = res_df.pivot_table(
    index=['Dataset', 'Explanation Method'],
    columns=['Shots', 'Percentage', "Loss"],
    values='iAUC'
).sort_index(axis=1)

perc_map = {
    False: "IAUC@",
    True: "mIoULoss@"
}

# Flatten the MultiIndex columns
# Collapse last two levels into one
res_df.columns = pd.MultiIndex.from_tuples([
    (lv0, f"{perc_map[lv2]}{lv1}") for lv0, lv1, lv2 in res_df.columns
])


# Sort by Model and then by Explanation Method using the provided orderings
res_df = res_df.sort_values(
    by=['Dataset', 'Explanation Method'],  # this ensures level names exist and are used
    key=lambda x: x.map(dataset_order) if x.name == 'Dataset' else x.map(method_order)
)

iauc_p_df = (res_df*100).round(2)

iauc_p_df

/tmp/ipykernel_2107814/549869128.py:21: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  res_df["Loss"] = res_df["Loss"].fillna(False)


1                              \
                                IAUC@0.01 mIoULoss@0.01 mIoULoss@0.05   
Dataset   Explanation Method                                            
Deepglobe Random                    47.29         14.12         22.21   
          Gaussian Noise Mask       46.49         21.19         27.34   
          Saliency                  47.65         15.03         21.50   
          Integrated Gradients      46.27         17.77         23.28   
          Blur IG                   46.75         16.36         24.02   
          XRAI                      47.96         13.02         13.16   
          Deep Lift                   NaN         16.65         23.65   
          LIME                      48.69         13.05          7.86   
          Unmasked AffEx (ours)     45.79         17.42         17.56   
          Masked AffEx (ours)       45.61         17.86         18.24   
          Signed AffEx (ours)       45.71         17.36         17.64   
ISIC      Random                    57.90         20.18         19.63   
          Gaussian Noise Mask       61.35         13.60          8.53   
          Saliency                  60.84         15.26          8.55   
          Integrated Gradients      59.40         16.43         13.45   
          Blur IG                   59.78         15.19         10.37   
          XRAI                      59.01         16.21          8.97   
          Deep Lift                   NaN         15.19         12.73   
          LIME                      60.11         15.94          9.16   
          Unmasked AffEx (ours)     58.49         10.12          7.08   
          Masked AffEx (ours)       58.52         10.34          7.06   
          Signed AffEx (ours)       58.51          9.56          6.50   
Lung      Random                    55.73         31.22         51.90   
          Gaussian Noise Mask       65.90         24.68         25.11   
          Saliency                  59.75         27.00         27.60   
          Integrated Gradients      60.11         27.47         30.35   
          Blur IG                   60.79         26.59         29.28   
          XRAI                      61.46         29.11         28.75   
          Deep Lift                   NaN         27.34         30.22   
          LIME                      62.98         27.20         27.66   
          Unmasked AffEx (ours)     63.30         26.36         28.32   
          Masked AffEx (ours)       63.26         26.27         28.28   
          Signed AffEx (ours)       63.32         25.91         27.88   

                                        5                              
                                IAUC@0.01 mIoULoss@0.01 mIoULoss@0.05  
Dataset   Explanation Method                                           
Deepglobe Random                    42.85         14.89         27.60  
          Gaussian Noise Mask       39.65         25.07         33.61  
          Saliency                  44.84         12.25         11.90  
          Integrated Gradients      42.85         14.92         22.15  
          Blur IG                   43.47         14.52         22.57  
          XRAI                      45.26         10.70         11.52  
          Deep Lift                   NaN           NaN           NaN  
          LIME                      48.32         10.24         11.08  
          Unmasked AffEx (ours)     43.52         12.94         13.10  
          Masked AffEx (ours)       43.42         13.25         13.87  
          Signed AffEx (ours)       43.55         12.92         13.33  
ISIC      Random                    70.72         22.49         18.71  
          Gaussian Noise Mask       78.46         20.89         17.32  
          Saliency                  78.46         20.61         15.49  
          Integrated Gradients      76.86         21.88         16.89  
          Blur IG                   76.15         20.28         15.09  
          XRAI                      75.76  

---

In [16]:
# joining iauc_dauc_df and iauc_p_df on index
final = iauc_dauc_df.join(iauc_p_df)
final = final.sort_index(level=0, axis=1)

metric_order = {
    'IAUC': 0,
    'DAUC': 1,
    'Diff.': 2,
    'IAUC@0.01': 3,
    'IAUC@0.05': 4,
    'mIoULoss@0.01': 5,
    'mIoULoss@0.05': 6,
}

new_cols  = sorted(final.columns, key=lambda x: (-x[0], metric_order[x[1]]))
final = final[new_cols]
final

5                                  1  \
                                IAUC@0.01 mIoULoss@0.01 mIoULoss@0.05   IAUC   
Dataset   Explanation Method                                                   
Deepglobe Random                    42.85         14.89         27.60  15.76   
          Gaussian Noise Mask       39.65         25.07         33.61  23.17   
          Saliency                  44.84         12.25         11.90  22.59   
          Integrated Gradients      42.85         14.92         22.15  21.11   
          Blur IG                   43.47         14.52         22.57  20.70   
          XRAI                      45.26         10.70         11.52  28.79   
          LIME                      48.32         10.24         11.08  31.58   
          Unmasked AffEx (ours)     43.52         12.94         13.10  29.11   
          Masked AffEx (ours)       43.42         13.25         13.87  29.16   
          Signed AffEx (ours)       43.55         12.92         13.33  27.96   
ISIC      Random                    70.72         22.49         18.71  42.05   
          Gaussian Noise Mask       78.46         20.89         17.32  46.17   
          Saliency                  78.46         20.61         15.49  45.09   
          Integrated Gradients      76.86         21.88         16.89  42.27   
          Blur IG                   76.15         20.28         15.09  44.48   
          XRAI                      75.76         20.98         14.45  43.25   
          LIME                      80.70         21.85         13.88  41.81   
          Unmasked AffEx (ours)     78.20         18.08         12.47  42.90   
          Masked AffEx (ours)       78.17         18.00         12.06  45.99   
          Signed AffEx (ours)       78.43         17.85         13.48  43.13   
Lung      Random                    62.89         27.76         56.42  43.72   
          Gaussian Noise Mask       84.28         25.42         26.01  42.63   
          Saliency                  72.63         26.15         24.71  44.64   
          Integrated Gradients      74.01         25.23         24.97  42.07   
          Blur IG                   73.52         25.46         25.55  47.27   
          XRAI                      75.51         29.37         27.94  46.79   
          LIME                      78.32         27.10         27.39  47.42   
          Unmasked AffEx (ours)     79.59         27.01         26.52  48.58   
          Masked AffEx (ours)       79.53         26.90         26.44  44.67   
          Signed AffEx (ours)       79.80         26.95         25.97  47.69   

                                                                       \
                                  DAUC  Diff. IAUC@0.01 mIoULoss@0.01   
Dataset   Explanation Method                                            
Deepglobe Random                 15.11   0.65     47.29         14.12   
          Gaussian Noise Mask    20.51   2.66     46.49         21.19   
          Saliency               18.33   4.26     47.65         15.03   
          Integrated Gradients   10.67  10.44     46.27         17.77   
          Blur IG                19.62   1.07     46.75         16.36   
          XRAI                   23.67   5.12     47.96         13.02   
          LIME                   19.63  11.95     48.69         13.05   
          Unmasked AffEx (ours)  22.06   7.05     45.79         17.42   
          Masked AffEx (ours)    23.71   5.45     45.61         17.86   
          Signed AffEx (ours)    21.35   6.61     45.71         17.36   
ISIC      Random                 42.18  -0.13     57.90         20.18   
          Gaussian Noise Mask    29.87  16.30     61.35         13.60   
          Saliency               33.85  11.24     60.84         15.26   
          Integrated Gradients   39.77   2.50     59.40         16.43   
          Blur IG                35.23   9.25     59.78         15.19   
          XRAI                   32.04  11.21     59.01         16.21   
          LI

In [22]:
directions = {
    (1,'IAUC'): None,
    (1,'DAUC'): None,
    (1,'Diff.'): 'max',
    (1,'IAUC@0.01'): 'max',
    (1,'mIoULoss@0.01'): 'min',
    (1,'mIoULoss@0.05'): 'min',
    (5,'IAUC'): None,
    (5,'DAUC'): None,
    (5,'Diff.'): 'max',
    (5,'IAUC@0.05'): 'max',
    (5,'mIoULoss@0.01'): 'min',
    (5,'mIoULoss@0.05'): 'min',
}

# -----------------------------
# 3) Compute bold winners only (no “Best” row)
# -----------------------------
def compute_best_mask(df: pd.DataFrame, directions: dict) -> pd.DataFrame:
    mask = pd.DataFrame(False, index=df.index, columns=df.columns)
    for model, group in df.groupby(level=0, sort=False):
        for col in df.columns:
            how = directions.get(col, None)
            if how is None:
                continue
            if how == 'max':
                best_val = group[col].max()
            elif how == 'min':
                best_val = group[col].min()
            else:
                continue
            mask.loc[group.index, col] = group[col] == best_val
    return mask

def latexify_bold(df: pd.DataFrame, directions: dict, floatfmt="{:.2f}"):
    """
    Return a LaTeX-ready DataFrame (as strings) where the best values are wrapped in \textbf{...}.
    """
    mask = compute_best_mask(df, directions)
    out = df.copy()
    for col in df.columns:
        for idx in df.index:
            val = df.at[idx, col]
            if pd.isna(val):
                out.at[idx, col] = ""
                continue
            s = floatfmt.format(val)
            if mask.at[idx, col]:
                s = f"\\textbf{{{s}}}"
            out.at[idx, col] = s
    return out.astype(str)

# -----------------------------
# 4) Convert to LaTeX
# -----------------------------
df_ltx = latexify_bold(final, directions, floatfmt="{:.2f}")

# Simple column alignment: first two left, rest right
colfmt = "ll" + "r"*df_ltx.shape[1]

latex_table = df_ltx.to_latex(
    escape=False,
    multirow=True,
    multicolumn=True,
    column_format=colfmt,
    bold_rows=False
)

print(latex_table)


\begin{tabular}{llrrrrrrrrr}
\toprule
 &  & \multicolumn{3}{r}{5} & \multicolumn{6}{r}{1} \\
 &  & IAUC@0.01 & mIoULoss@0.01 & mIoULoss@0.05 & IAUC & DAUC & Diff. & IAUC@0.01 & mIoULoss@0.01 & mIoULoss@0.05 \\
Dataset & Explanation Method &  &  &  &  &  &  &  &  &  \\
\midrule
\multirow[t]{10}{*}{Deepglobe} & Random & 42.85 & 14.89 & 27.60 & 15.76 & 15.11 & 0.65 & 47.29 & 14.12 & 22.21 \\
 & Gaussian Noise Mask & 39.65 & 25.07 & 33.61 & 23.17 & 20.51 & 2.66 & 46.49 & 21.19 & 27.34 \\
 & Saliency & 44.84 & 12.25 & 11.90 & 22.59 & 18.33 & 4.26 & 47.65 & 15.03 & 21.50 \\
 & Integrated Gradients & 42.85 & 14.92 & 22.15 & 21.11 & 10.67 & 10.44 & 46.27 & 17.77 & 23.28 \\
 & Blur IG & 43.47 & 14.52 & 22.57 & 20.70 & 19.62 & 1.07 & 46.75 & 16.36 & 24.02 \\
 & XRAI & 45.26 & 10.70 & 11.52 & 28.79 & 23.67 & 5.12 & 47.96 & \textbf{13.02} & 13.16 \\
 & LIME & 48.32 & \textbf{10.24} & \textbf{11.08} & 31.58 & 19.63 & \textbf{11.95} & \textbf{48.69} & 13.05 & \textbf{7.86} \\
 & Unmasked AffEx (ours

/tmp/ipykernel_2107814/1375683202.py:50: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '42.85' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  out.at[idx, col] = s
/tmp/ipykernel_2107814/1375683202.py:50: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '14.89' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  out.at[idx, col] = s
/tmp/ipykernel_2107814/1375683202.py:50: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '27.60' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  out.at[idx, col] = s
/tmp/ipykernel_2107814/1375683202.py:50: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error i

In [20]:
final.columns

MultiIndex([(5,     'IAUC@0.01'),
            (5, 'mIoULoss@0.01'),
            (5, 'mIoULoss@0.05'),
            (1,          'IAUC'),
            (1,          'DAUC'),
            (1,         'Diff.'),
            (1,     'IAUC@0.01'),
            (1, 'mIoULoss@0.01'),
            (1, 'mIoULoss@0.05')],
           )